In [53]:
from env import config, BASE_DIR
from sqlalchemy import create_engine, text

In [54]:
DATABASE_URL = config("FLIGHT_PRICES_DATABASE_URL", cast=str, default=None)
DATABASE_URL is not None

True

In [55]:
engine = create_engine(str(DATABASE_URL))

In [56]:
DATASET_DIR = BASE_DIR / "datasets"
DATASET_DIR.exists()

True

In [57]:
FLIGHT_PRICES_CSV = DATASET_DIR / "itineraries-min-50k.csv"
FLIGHT_PRICES_CSV.exists()

True

In [58]:
import pandas as pd

In [59]:
df = pd.read_csv(str(FLIGHT_PRICES_CSV))
df.head(n=10)

,legId,searchDate,flightDate,startingAirport,destinationAirport,fareBasisCode,travelDuration,elapsedDays,isBasicEconomy,isRefundable,...,segmentsArrivalTimeEpochSeconds,segmentsArrivalTimeRaw,segmentsArrivalAirportCode,segmentsDepartureAirportCode,segmentsAirlineName,segmentsAirlineCode,segmentsEquipmentDescription,segmentsDurationInSeconds,segmentsDistance,segmentsCabinCode
0,9ca0e81111c683bec1012473feefd28f,2022-04-16,2022-04-17,ATL,BOS,LA0NX0MC,PT2H29M,0.0,False,False,...,1650223560,2022-04-17T15:26:00.000-04:00,BOS,ATL,Delta,DL,Airbus A321,8940,947,coach
1,98685953630e772a098941b71906592b,2022-04-16,2022-04-17,ATL,BOS,LA0NX0MC,PT2H30M,0.0,False,False,...,1650200400,2022-04-17T09:00:00.000-04:00,BOS,ATL,Delta,DL,Airbus A321,9000,947,coach
2,98d90cbc32bfbb05c2fc32897c7c1087,2022-04-16,2022-04-17,ATL,BOS,LA0NX0MC,PT2H30M,0.0,False,False,...,1650218700,2022-04-17T14:05:00.000-04:00,BOS,ATL,Delta,DL,Boeing 757-200,9000,947,coach
3,969a269d38eae583f455486fa90877b4,2022-04-16,2022-04-17,ATL,BOS,LA0NX0MC,PT2H32M,0.0,False,False,...,1650227460,2022-04-17T16:31:00.000-04:00,BOS,ATL,Delta,DL,Airbus A321,9120,947,coach
4,980370cf27c89b40d2833a1d5afc9751,2022-04-16,2022-04-17,ATL,BOS,LA0NX0MC,PT2H34M,0.0,False,False,...,1650213180,2022-04-17T12:33:00.000-04:00,BOS,ATL,Delta,DL,Airbus A321,9240,947,coach
5,79eda9f841e226a1e2121d74211e595c,2022-04-16,2022-04-17,ATL,BOS,VH0AUEL1,PT2H38M,0.0,False,False,...,1650216180,2022-04-17T13:23:00.000-04:00,BOS,ATL,JetBlue Airways,B6,NaN,9480,947,coach
6,9335fae376c38bb61263281779f469ec,2022-04-16,2022-04-17,ATL,BOS,V0AJZNN1,PT4H12M,0.0,False,False,...,1650203400||1650213120,2022-04-17T09:50:00.000-04:00||2022-04-17T12:3...,CLT||BOS,ATL||CLT,American Airlines||American Airlines,AA||AA,Airbus A320||Airbus A320,5400||7500,228||728,coach||coach
7,3904bf87f2d1daf334f1ae7e3b876028,2022-04-16,2022-04-17,ATL,BOS,V0AJZNN1,PT5H18M,0.0,False,False,...,1650203400||1650217080,2022-04-17T09:50:00.000-04:00||2022-04-17T13:3...,CLT||BOS,ATL||CLT,American Airlines||American Airlines,AA||AA,Airbus A320||Boeing 737-800,5400||8280,228||728,coach||coach
8,d93988734c44a3c075d9efe373352507,2022-04-16,2022-04-17,ATL,BOS,V0AJZNN1,PT5H32M,0.0,False,False,...,1650198060||1650213120,2022-04-17T08:21:00.000-04:00||2022-04-17T12:3...,CLT||BOS,ATL||CLT,American Airlines||American Airlines,AA||AA,Airbus A319||Airbus A320,4860||7500,228||728,coach||coach
9,562e7d5dd6ecbf1509c0c19711dbdca9,2022-04-16,2022-04-17,ATL,BOS,V0AJZNN1,PT6H38M,0.0,False,False,...,1650198060||1650217080,2022-04-17T08:21:00.000-04:00||2022-04-17T13:3...,CLT||BOS,ATL||CLT,American Airlines||American Airlines,AA||AA,Airbus A319||Boeing 737-800,4860||8280,228||728,coach||coach


In [60]:
startingAirports = set(df.startingAirport.unique().tolist())

In [61]:
destinationAirports = set(df.destinationAirport.unique().tolist())

In [62]:
missing = startingAirports - destinationAirports
missing

set()

In [26]:
#%pip install -U airportsdata

In [63]:
import airportsdata
airports = airportsdata.load('IATA')

In [64]:
available_airports = [airports[x] for x in startingAirports]
available_airports

[{'icao': 'KATL',
  'iata': 'ATL',
  'name': 'Hartsfield - Jackson Atlanta International Airport',
  'city': 'Atlanta',
  'subd': 'Georgia',
  'country': 'US',
  'elevation': 1026.2,
  'lat': 33.6367,
  'lon': -84.42786,
  'tz': 'America/New_York',
  'lid': 'ATL'},
 {'icao': 'KORD',
  'iata': 'ORD',
  'name': "Chicago O'Hare International Airport",
  'city': 'Chicago',
  'subd': 'Illinois',
  'country': 'US',
  'elevation': 680.0,
  'lat': 41.97694,
  'lon': -87.90815,
  'tz': 'America/Chicago',
  'lid': 'ORD'},
 {'icao': 'KSFO',
  'iata': 'SFO',
  'name': 'San Francisco International Airport',
  'city': 'San Francisco',
  'subd': 'California',
  'country': 'US',
  'elevation': 13.1,
  'lat': 37.61881,
  'lon': -122.37542,
  'tz': 'America/Los_Angeles',
  'lid': 'SFO'},
 {'icao': 'KIAD',
  'iata': 'IAD',
  'name': 'Washington Dulles International Airport',
  'city': 'Washington',
  'subd': 'Dist. Of Columbia',
  'country': 'US',
  'elevation': 312.3,
  'lat': 38.9,
  'lon': -77.5,
  't

In [65]:
airports_df = pd.DataFrame(available_airports)

In [66]:
airports_df.dtypes

icao          object
iata          object
name          object
city          object
subd          object
country       object
elevation    float64
lat          float64
lon          float64
tz            object
lid           object
dtype: object

In [67]:
table_name = "airports"
airports_df.to_sql(table_name, engine, if_exists='replace', index=True)

16

In [68]:
from sqlalchemy import create_engine, Column, BigInteger, String, Float
from sqlalchemy.orm import sessionmaker, DeclarativeBase

class Base(DeclarativeBase):
    pass

class Airport(Base):
    __tablename__ = "airports"
    __table_args__ = {"extend_existing": True}
    id = Column("index", BigInteger, primary_key = True)
    name = Column(String)
    icao = Column(String)
    iata = Column(String)
    city = Column(String)
    subd = Column(String)
    country = Column(String)
    elevation = Column(Float)
    lat = Column(Float)
    lon = Column(Float)
    tz = Column(String)
    lid = Column(String)

In [69]:
from pydantic import BaseModel, Field

class AirportSchema(BaseModel):
    id: int
    label: str = Field(alias = "name")
    value: str = Field(alias = "iata")

    class Config:
        from_attributes = True

In [70]:
SessionLocal = sessionmaker(bind=engine)
session = SessionLocal()

queryset = session.query(Airport).limit(10).all()
for row_value in queryset:
    data = AirportSchema.model_validate(row_value)
    print(data.model_dump_json())

session.close()

{"id":0,"label":"Hartsfield - Jackson Atlanta International Airport","value":"ATL"}
{"id":1,"label":"Chicago O'Hare International Airport","value":"ORD"}
{"id":2,"label":"San Francisco International Airport","value":"SFO"}
{"id":3,"label":"Washington Dulles International Airport","value":"IAD"}
{"id":4,"label":"John F Kennedy International Airport","value":"JFK"}
{"id":5,"label":"Dallas-Fort Worth International Airport","value":"DFW"}
{"id":6,"label":"Los Angeles International Airport","value":"LAX"}
{"id":7,"label":"Laguardia Airport","value":"LGA"}
{"id":8,"label":"Detroit Metro Wayne County Airport","value":"DTW"}
{"id":9,"label":"Newark Liberty International Airport","value":"EWR"}
